# OmniVision3D — YOLOv8 Training on Google Colab

**Target:** DJI Mini 4 Pro detection  
**Model:** YOLOv8 nano  
**Dataset:** 1,800 synthetic renders (RGB sky-composited)  

### Instructions
1. Set runtime to **GPU** (Runtime → Change runtime type → T4 GPU)
2. Run cells in order
3. Cell 2 will prompt you to upload `yolo_dataset.zip`
4. Training takes ~10–15 min on T4
5. Cell 6 downloads `best.pt` and `best.onnx` automatically

In [ ]:
# Cell 1 — Setup
!pip install ultralytics -q
from ultralytics import YOLO
import os, torch
print(f'PyTorch: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Ready')

In [ ]:
# Cell 2 — Upload dataset
# Upload yolo_dataset.zip when prompted
from google.colab import files
import zipfile, os

uploaded = files.upload()  # select yolo_dataset.zip

print('Extracting...')
with zipfile.ZipFile('yolo_dataset.zip', 'r') as z:
    z.extractall('.')

for split in ['train', 'val', 'test']:
    imgs = len(os.listdir(f'yolo_dataset/images/{split}'))
    lbls = len(os.listdir(f'yolo_dataset/labels/{split}'))
    print(f'{split}: {imgs} images, {lbls} labels')

In [ ]:
# Cell 3 — Fix dataset.yaml paths for Colab
# Colab runs from /content — paths must be absolute
yaml_content = """path: /content/yolo_dataset
train: images/train
val: images/val
test: images/test
nc: 1
names: [dji_mini_4_pro]
"""
with open('yolo_dataset/dataset.yaml', 'w') as f:
    f.write(yaml_content)
print('dataset.yaml updated for Colab paths')
print(open('yolo_dataset/dataset.yaml').read())

In [ ]:
# Cell 4 — Train YOLOv8 nano
# T4 GPU: ~10-15 min for 50 epochs
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data='yolo_dataset/dataset.yaml',
    epochs=50,
    imgsz=512,
    batch=16,
    name='dji_detector',
    project='checkpoints/yolo',
    patience=10,
    device=0,
    workers=4,
    pretrained=True,
    optimizer='Adam',
    lr0=0.001,
    augment=True,
    fliplr=0.5,
    translate=0.2,
    scale=0.3,
    degrees=15.0,
    mosaic=1.0
)

print(f'Training complete')
print(f'Best model: {results.save_dir}/weights/best.pt')

In [ ]:
# Cell 5 — Evaluate on test set
metrics = model.val(
    data='yolo_dataset/dataset.yaml',
    split='test'
)

print(f'mAP50:     {metrics.box.map50:.3f}')
print(f'mAP50-95:  {metrics.box.map:.3f}')
print(f'Precision: {metrics.box.p[0]:.3f}')
print(f'Recall:    {metrics.box.r[0]:.3f}')
print()
print('Target thresholds:')
print(f'  mAP50 > 0.85:     {"PASS" if metrics.box.map50 > 0.85 else "FAIL"}')
print(f'  Precision > 0.85: {"PASS" if metrics.box.p[0] > 0.85 else "FAIL"}')
print(f'  Recall > 0.90:    {"PASS" if metrics.box.r[0] > 0.90 else "FAIL"}')

In [ ]:
# Cell 6 — Export to ONNX and download both weights
import glob
from google.colab import files

# Export best.pt to ONNX
best_pt = glob.glob('checkpoints/yolo/**/best.pt', recursive=True)[0]
export_model = YOLO(best_pt)
export_model.export(format='onnx', imgsz=512, simplify=True)

best_onnx = glob.glob('checkpoints/yolo/**/best.onnx', recursive=True)[0]

pt_size   = os.path.getsize(best_pt)   / 1024 / 1024
onnx_size = os.path.getsize(best_onnx) / 1024 / 1024
print(f'PT model:   {best_pt} ({pt_size:.1f} MB)')
print(f'ONNX model: {best_onnx} ({onnx_size:.1f} MB)')
print()
print('Downloading...')
files.download(best_pt)
files.download(best_onnx)